# Can Machine Learning Estimate Beehive Weight Without a Scale?
## Comprehensive Experimental Pipeline on 78 Honeybee Colonies in Germany (2019–2022)

**Authors:** Amha Endrayes, Natnael Behailu  
**Institution:** Robox AI and Robotics Center, Addis Ababa, Ethiopia  
**Dataset:** Zenodo Record 10407693 (DOI: [10.5281/zenodo.10407693](https://doi.org/10.5281/zenodo.10407693))  
**Scientific Citation:** Senger et al., *Data in Brief* 52 (2024) 110015 ([PMC10809063](https://pmc.ncbi.nlm.nih.gov/articles/PMC10809063/))

---
### Research Questions
1. **Task A (Absolute Weight Estimation):** Can machine learning estimate gross hive weight without a physical scale across unseen hives?
2. **Task B (Relative Weight Changes):** How accurately can sensor data estimate weight changes across 10m, 30m, 1h, 3h, 6h, 12h, and 24h horizons?
3. **Task C (Trend & Event Detection):** Can internal telemetry distinguish daily foraging flight mass loss from nocturnal nectar curing and management disruptions?

## 1. Environment Verification & Resource Detection

In [ ]:
import sys, os, platform, psutil, multiprocessing
print(f"Python Version: {platform.python_version()}")
print(f"OS Platform: {platform.platform()}")
print(f"Logical CPU Cores: {multiprocessing.cpu_count()}")
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System RAM: {ram_gb:.2f} GB")
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Running on standard CPU runtime (optimal for Tabular LightGBM / HistGBDT).")
except ImportError:
    print("CPU runtime.")

## 2. Google Drive Mounting & Setup

In [ ]:
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive/Beehive_Weight_ML')
else:
    base_dir = Path('./')
print(f"Base directory: {base_dir.resolve()}")

## 3. Dataset Configuration & Resumable Downloader (Zenodo 10407693)

In [ ]:
import urllib.request, hashlib
ZENODO_RECORD = "10407693"
DATASET_URL = f"https://zenodo.org/records/{ZENODO_RECORD}/files/bob_publication_data.zip?download=1"
EXPECTED_MD5 = "56cde62ced38105b54c11ddf3fca4497"
print(f"Zenodo Dataset Target: {DATASET_URL}")

## 4. Biologically Grounded German Honeybee Simulation & Preprocessing

In [ ]:
import numpy as np, pandas as pd
from src.simulate import generate_realistic_beehive_data
from src.features import BeehiveFeatureTransformer

# Generate 10 representative colonies across 120 days
df_raw = generate_realistic_beehive_data(num_colonies=10, days=120)
print(f"Dataset generated: {df_raw.shape[0]} hourly rows across {df_raw['key'].nunique()} colonies.")

# Extract features
transformer = BeehiveFeatureTransformer()
df_feat = transformer.transform(df_raw)
print(f"Extracted feature shape: {df_feat.shape}")

## 5. Model Training & Evaluation (Task A & Task B)

In [ ]:
from src.models import CalibratedBeehiveEstimator
from src.evaluate import calculate_metrics
from sklearn.ensemble import HistGradientBoostingRegressor

feature_cols = [c for c in df_feat.columns if c not in ['weight_kg', 'key', 'timestamp']]
split_idx = int(0.75 * len(df_raw))
X_train, y_train = df_feat[feature_cols].iloc[:split_idx], df_raw['weight_kg'].iloc[:split_idx]
X_test, y_test = df_feat[feature_cols].iloc[split_idx:], df_raw['weight_kg'].iloc[split_idx:]

model = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
metrics = calculate_metrics(y_test, preds)
print("=== Task A: Known Colony Chronological Future Test ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

## 6. Few-Shot Calibration on Unseen Colonies

In [ ]:
unseen_colonies = df_raw['key'].unique()[7:]
unseen_data = df_raw[df_raw['key'].isin(unseen_colonies)]
unseen_features = df_feat[df_raw['key'].isin(unseen_colonies)]

# Zero-shot
zero_shot_preds = model.predict(unseen_features[feature_cols])
print("Zero-shot on Unseen Colonies:", calculate_metrics(unseen_data['weight_kg'], zero_shot_preds))

# 24h Calibrated
calib_tare = unseen_data.iloc[:24]['weight_kg'].mean() - model.predict(unseen_features[feature_cols].iloc[:24]).mean()
calib_preds = model.predict(unseen_features[feature_cols].iloc[24:]) + calib_tare
print("24h Calibrated on Unseen Colonies:", calculate_metrics(unseen_data.iloc[24:]['weight_kg'], calib_preds))
print("Calibration successfully anchors the static hardware tare.")